# 08 · Recursion with matrices and vectors / Recursión con matrices y vectores

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/08-recursion-with-matrices.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#2563eb,rgba(37,99,235,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#2563eb">PART IV · DEMO · 10 MIN</span>

## Practise today / Practica hoy

Explain a matrix state update and connect repeated updates with a matrix power.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Explicar una actualización de estado matricial y relacionar su repetición con una potencia de matriz.</div></div>

## Explore later / Explora después

Use power iteration and evaluate recursive forecasts on airline traffic.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 0">Usar iteración de potencias y evaluar pronósticos recursivos del tráfico aéreo.</div></div>

Follow the core block immediately below. / Sigue el bloque esencial de abajo.

<!-- CORE-PATH -->
## Core path / Ruta esencial

Read and run this block from top to bottom: **recall → example → attempt → feedback → checkpoint**. Preparation and feedback definitions appear where needed. Try before opening a folded solution. Stop at **Core complete**; everything after it is **Explore later**.

🇪🇸 Lee y ejecuta este bloque de arriba abajo: **recuerda → ejemplo → intento → retroalimentación → comprobación**. La preparación y las funciones de comprobación aparecen donde se necesitan. Inténtalo antes de abrir una solución plegada. Detente en **Fin de la ruta esencial**; después empieza **Explora después**.

### Recall / Recuerda

Recall notebook 07: if two inputs become the same output, can an update always be reversed? Keep this question in mind as states repeat.

🇪🇸 Recuerda el cuaderno 07: si dos entradas producen la misma salida, ¿siempre puede invertirse una actualización? Tenlo presente al repetir estados.

## Setup / Preparación

Run this cell first.

It loads the real monthly airline-passenger dataset used later and the small visualization tools used throughout the notebook.

> 🇪🇸 Ejecuta primero esta celda.
>
> Carga el conjunto real de pasajeros mensuales de aerolíneas que usaremos más adelante y las herramientas de visualización del cuaderno.

### Core prep 1/1 · Preparación esencial

Run the next cell. / Ejecuta la siguiente celda.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

FLIGHTS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/flights.csv"
)

flights = pd.read_csv(FLIGHTS)

y = flights["passengers"].to_numpy(float)

labels = (
    flights["year"].astype(str)
    + "-"
    + flights["month"].astype(str).str[:3]
).to_numpy()

rng = np.random.default_rng(0)

print("Real months / Meses reales:", len(y))
print("Range / Periodo:", labels[0], "→", labels[-1])
print("Passengers min/max / Pasajeros mín/máx:", int(y.min()), int(y.max()))
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## One rule, applied again / Una regla, aplicada otra vez

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#2563eb,rgba(37,99,235,0))"></div>

One 2×2 matrix applied to a pair of numbers, again and again, produces 1, 1, 2, 3, 5 — the Fibonacci numbers; the matrix never changes, the state does. That is four steps of <code>x[t+1] = F @ x[t]</code> from <code>[1, 0]</code>.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-08-recurrence.gif" alt="An animation of four steps of a matrix recurrence. A fixed 2 by 2 matrix multiplies a 2-element state vector to give the next state, and the states run through the Fibonacci numbers. The matrix is identical in every frame." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una matriz 2×2 aplicada a un par de números, una y otra vez, produce 1, 1, 2, 3, 5 — los números de Fibonacci; la matriz nunca cambia, el estado sí. Son cuatro pasos de <code>x[t+1] = F @ x[t]</code> desde <code>[1, 0]</code>.</div>


## Three useful words / Tres palabras útiles

The picture above is this rule with `A = F`.

🇪🇸 La imagen de arriba es esta regla con `A = F`.

| Term / Término | Plain meaning / Significado sencillo |
|---|---|
| **State / Estado** | the information we carry from one step to the next / la información que llevamos de un paso al siguiente |
| **Update rule / Regla de actualización** | the recipe that converts the current state into the next one / la receta que convierte el estado actual en el siguiente |
| **Recursion / Recurrence / Recursión / Recurrencia** | applying that update repeatedly / aplicar esa actualización repetidamente |

In this notebook, the update rule often looks like:

`x[t+1] = A @ x[t]`

Read it as:

> **next state = same matrix × current state**

> 🇪🇸 Léelo como:
>
> **estado siguiente = la misma matriz × estado actual**


## Read the recurrence as a sentence / Lee la recurrencia como una frase

For:

`x[t+1] = A @ x[t]`

- `t` = current step / paso actual
- `x[t]` = current state / estado actual
- `A` = update rule written as a matrix / regla de actualización escrita como matriz
- `x[t+1]` = next state / estado siguiente

The matrix `A` stays the same in the simplest examples. The state changes at every step.

> 🇪🇸 En los ejemplos más sencillos, la matriz `A` permanece igual. Lo que cambia en cada paso es el estado.

## Exercise 1 — Fibonacci as a matrix state / Ejercicio 1 — Fibonacci como estado matricial

Fibonacci is usually written:

`f[n+1] = f[n] + f[n-1]`


This looks like a scalar recurrence.

But the next value needs **two pieces of memory**:

- the current Fibonacci number;
- the previous Fibonacci number.

So we store both in one state:

`[f[n], f[n-1]]`

and update it with:

`F = [[1,1],[1,0]]`

$$
\begin{bmatrix} f_{n+1} \\ f_{n} \end{bmatrix}
 = 
\underbrace{\begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}}_{F}
\begin{bmatrix} f_{n} \\ f_{n-1} \end{bmatrix}
\qquad\Longrightarrow\qquad
x_{t} = F^{t} x_{0}
$$

Read it as: carrying two numbers instead of one turns the recurrence into a
single matrix applied over and over — and $t$ steps then cost one matrix power
rather than a loop of length $t$.

### Why does this work? / ¿Por qué funciona?

The first row says:

`new first value = current + previous`

The second row says:

`new second value = old current`

That means the state automatically shifts forward.

> 🇪🇸 La primera fila suma los dos valores para crear el siguiente Fibonacci.
>
> La segunda fila copia el valor actual a la posición de “anterior”.
>
> Así el estado avanza automáticamente.

### Optional hints / Pistas opcionales

Try first; open one hint at a time. / Inténtalo primero; abre una pista a la vez.

<details>
<summary>Hint 1 / Pista 1</summary>

Multiply F by a two-entry state on paper. Which old value must be remembered after the first entry changes?

🇪🇸 Multiplica F por un estado de dos entradas en papel. ¿Qué valor anterior debe recordarse al cambiar la primera entrada?

</details>

<details>
<summary>Hint 2 / Pista 2</summary>

Keep v0 unchanged and update v = F @ v once per loop. Ten updates produce eleven states if you include the initial state.

🇪🇸 Conserva v0 y actualiza v = F @ v una vez por vuelta. Diez actualizaciones producen once estados si incluyes el inicial.

</details>



### Core activity · Actividad esencial

**Predict → Run → Explain → Check**

1. **Predict.** The picture stopped after four steps; predict the state after ten.
2. **Run.** Complete Exercise 1.
3. **Explain.** Explain why both entries are needed.
4. **Check.** Compare the loop with matrix_power(F, 10) @ v0. Count updates, not printed states.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

1. **Predice.** La imagen se detuvo después de cuatro pasos; predice el estado después de diez.
2. **Ejecuta.** Completa el Ejercicio 1.
3. **Explica.** Explica por qué se necesitan ambas entradas.
4. **Comprueba.** Compara el bucle con matrix_power(F, 10) @ v0. Cuenta actualizaciones, no estados impresos.

</details>

Prediction / Predicción: ___  
Evidence / Evidencia: ___  
Revised explanation / Explicación revisada: ___


In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Define:
#       F = [[1, 1],
#            [1, 0]]
# 2. Start with state [1, 0].
# 3. Apply F repeatedly for 10 steps.
# 4. Print every state.
# 5. Compare the final result with:
#       np.linalg.matrix_power(F, 10) @ v0
#
# ES:
# 1. Define:
#       F = [[1, 1],
#            [1, 0]]
# 2. Empieza con el estado [1, 0].
# 3. Aplica F repetidamente durante 10 pasos.
# 4. Imprime cada estado.
# 5. Compara el resultado final con:
#       np.linalg.matrix_power(F, 10) @ v0

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

F = np.array(
    [[1, 1],
     [1, 0]],
    dtype=object,
)

v0 = np.array([1, 0], dtype=object)

states = [v0.copy()]
v = v0.copy()

for _ in range(10):
    v = F @ v
    states.append(v.copy())

via_power = np.linalg.matrix_power(F, 10) @ v0

print("Step / Paso | State / Estado")
for k, state in enumerate(states):
    print(f"{k:>4}        {state.tolist()}")

print()
print("Loop final / Final del ciclo:", v.tolist())
print("matrix_power:", via_power.tolist())
print("Same result / Mismo resultado:", np.array_equal(v, via_power))
print("Fibonacci(10):", int(v[1]))
print()
print("EN: matrix_power compresses ten repeated updates into one matrix power.")
print("ES: matrix_power comprime diez actualizaciones repetidas en una sola potencia matricial.")

<details>
<summary><strong>What did Exercise 1 teach? / ¿Qué enseñó el Ejercicio 1?</strong></summary>

A recurrence may look like a sequence of scalar formulas, but we can often collect the necessary memory into a **state vector**.

Then one repeated matrix multiplication advances the whole state.

`matrix_power(F, n)` is useful because:

`Fⁿ @ x[0]`

represents applying the same state update `n` times.

> 🇪🇸 Una recurrencia puede parecer una secuencia de fórmulas escalares, pero podemos reunir la memoria necesaria en un **vector de estado**.
>
> `Fⁿ @ x[0]` representa aplicar la misma actualización `n` veces.

</details>

### Checkpoint / Comprobación

For `F = [[1, 1], [1, 0]]` and state `[3, 2]`, predict the next state and explain what each entry stores.

🇪🇸 Para `F = [[1, 1], [1, 0]]` y estado `[3, 2]`, predice el estado siguiente y explica qué guarda cada entrada.

Answer / Respuesta: ___

<details>
<summary>Check after attempting / Comprueba después de intentarlo</summary>

The next state is `[5, 3]`: the next Fibonacci value and the previous current value. Two updates are `F @ F @ state`, or `matrix_power(F, 2) @ state`.

🇪🇸 El estado siguiente es `[5, 3]`: el siguiente valor de Fibonacci y el valor actual anterior. Dos actualizaciones son `F @ F @ state` o `matrix_power(F, 2) @ state`.

</details>

## Core complete / Fin de la ruta esencial

Keep your prediction, evidence, and explanation. Follow the facilitator’s quiz and break schedule before continuing.

🇪🇸 Guarda tu predicción, evidencia y explicación. Sigue las pausas y quizzes del facilitador antes de continuar.

[Next: Notebook 09 / Siguiente: cuaderno 09](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-matrix-factorizations.ipynb).

## Explore later / Explora después

Optional reference, exercises, and explorers. These are outside this section’s live core. Continue in order when studying them; some reuse earlier setup.

🇪🇸 Material de consulta, ejercicios y exploradores opcionales. Quedan fuera de la ruta esencial en vivo. Continúa en orden al estudiarlos; algunos reutilizan la preparación anterior.

### The operator, not the loop / El operador, no el bucle

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#2563eb,rgba(37,99,235,0))"></div>

The same matrix multiplied by itself. `F` takes one step of the recurrence, so `F ** n` takes n of them in a single multiplication — and the Fibonacci numbers are already sitting in the entries, not produced by a loop.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-08-power.gif" alt="An animation showing the 2 by 2 matrix of 1, 1, 1, 0 raised to the first, second, third and fourth power, with 1, 2, 3 and 5 appearing in turn in the top left entry." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La misma matriz multiplicada por sí misma. <code>F</code> da un paso de la recurrencia, así que <code>F ** n</code> da n pasos en una sola multiplicación, y los números de Fibonacci ya están en las entradas: no los produce ningún bucle.</div>

### What repetition converges on / A qué converge la repetición

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#2563eb,rgba(37,99,235,0))"></div>

Apply the same matrix enough times and the result stops being about where you started. The ratio of the two entries is something you can watch settle without knowing what an eigenvalue is — 1, then 2, then 1.500, then 1.667 — and ten steps later it has stopped moving at 1.618. The last frame names it.

$$
x_t = F^{t} x_0
\qquad\qquad
\frac{x_t}{\lVert x_t \rVert} \longrightarrow v_1,
\quad\text{where}\quad F v_1 = \lambda_1 v_1
$$

Read it as: the direction converges even though the length does not. Whichever
eigenvalue is largest in absolute value is the one that survives repetition, so
a long enough run tells you about the matrix and almost nothing about where it
was started.

<img src="https://project-delphi.github.io/tensors-workshop/images/cube-08-direction.gif" alt="An animation of the Fibonacci matrix applied repeatedly to a starting vector, showing the four resulting 2-element states. The ratio of the two entries is then shown for each step, running 1, 2, 1.500, 1.667, and after ten more steps all four values read 1.618. The last frame shows the two eigenvalues of the matrix, 1.618 and minus 0.618, with the larger one highlighted." style="max-width:100%;display:block;margin:2.2em auto;border-radius:10px">

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:11px">🇪🇸 ESPAÑOL</div>Aplica la misma matriz suficientes veces y el resultado deja de depender de dónde empezaste. La razón entre las dos entradas se puede ver estabilizarse sin saber qué es un valor propio —1, luego 2, luego 1.500, luego 1.667— y diez pasos después ha dejado de moverse en 1.618. El último fotograma le pone nombre.</div>

In [ ]:
#@title ⏸️ Step through the animations / Recorre las animaciones { display-mode: 'form' }

# Plumbing, not a lesson. The animations above loop forever and a GIF cannot
# be paused — so this fetches the same frames and hands them over one at a
# time, at whatever pace you read at.
# Plomería, no una lección: trae los mismos fotogramas y los entrega de uno en
# uno, al ritmo al que leas.

import io
import urllib.request

import ipywidgets as widgets
from IPython.display import display
from PIL import Image

gif_urls = [
    "https://project-delphi.github.io/tensors-workshop/images/cube-08-recurrence.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-08-power.gif",
    "https://project-delphi.github.io/tensors-workshop/images/cube-08-direction.gif",
]


def gif_frames(url):
    """Every frame of an animated GIF, as PNG bytes."""
    with urllib.request.urlopen(url, timeout=30) as response:
        gif = Image.open(io.BytesIO(response.read()))
    out = []
    try:
        while True:
            buffer = io.BytesIO()
            gif.convert("RGB").save(buffer, format="PNG")
            out.append(buffer.getvalue())
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    return out


try:
    gif_cache = {url: gif_frames(url) for url in gif_urls}
except Exception as error:  # offline, or the site is down
    print("EN: could not reach the site, so there are no frames to step "
          "through.", error)
    print("ES: no se pudo acceder al sitio, así que no hay fotogramas que "
          "recorrer.", error)
else:
    gif_pick = widgets.Dropdown(
        options=[(url.rsplit("/", 1)[1], url) for url in gif_urls],
        description="Animation / Animación:",
        style={"description_width": "180px"},
    )
    gif_step = widgets.IntSlider(
        min=1, max=len(gif_cache[gif_urls[0]]), value=1,
        description="Frame / Fotograma:",
        style={"description_width": "180px"},
        continuous_update=False,
    )
    gif_prev = widgets.Button(description="◀ Prev")
    gif_next = widgets.Button(description="Next ▶")
    # An Image widget, deliberately, and never widgets.Output: a payload
    # leaving an Output widget makes nbclient wait out the whole cell timeout
    # (see scripts/test_notebooks.py). This one is a plain bytes trait.
    gif_view = widgets.Image(format="png",
                             layout=widgets.Layout(max_width="100%"))

    def gif_show(*_):
        frames = gif_cache[gif_pick.value]
        gif_step.max = len(frames)
        gif_view.value = frames[min(gif_step.value, len(frames)) - 1]

    def gif_bump(delta):
        def click(_):
            frames = gif_cache[gif_pick.value]
            gif_step.value = (gif_step.value - 1 + delta) % len(frames) + 1
        return click

    gif_prev.on_click(gif_bump(-1))
    gif_next.on_click(gif_bump(+1))
    gif_pick.observe(gif_show, names="value")
    gif_step.observe(gif_show, names="value")
    gif_show()

    display(widgets.VBox([
        gif_pick,
        widgets.HBox([gif_prev, gif_step, gif_next]),
        gif_view,
    ]))


## Repetition builds, or amplifies / La repetición construye o amplifica

The same recursive skeleton has three different consequences in this notebook. Fibonacci generates a sequence exactly. Power iteration reveals a dominant direction. Recursive forecasting feeds its own output back in, so an error travels forward.

Repetition either **builds structure** or **propagates error**, and the update rule is what decides which.

> 🇪🇸 El mismo esqueleto recursivo tiene tres consecuencias distintas aquí. Fibonacci genera una secuencia exacta. La iteración de potencias revela una dirección dominante. La predicción recursiva realimenta su propia salida, de modo que un error viaja hacia adelante.
>
> La repetición **construye estructura** o **propaga error**, y la regla de actualización decide cuál de las dos.


### Interactive state-update translator / Traductor interactivo de actualización de estado

Choose one example and identify its state and update rule.

> 🇪🇸 Elige un ejemplo e identifica su estado y su regla de actualización.

In [ ]:
#@title 🔁 State explorer / Explorador de estados — run me / ejecútame { display-mode: 'form' }

state_example = widgets.Dropdown(
    options=[
        ("Fibonacci", "fib"),
        ("Power iteration / Iteración de potencias", "power"),
        ("Recursive forecast / Pronóstico recursivo", "forecast"),
    ],
    value="fib",
    description="Example / Ejemplo:",
    style={"description_width": "120px"},
)

def explain_state_example(example):
    examples = {
        "fib": (
            "[f[n], f[n-1]]",
            "multiply by the same 2×2 Fibonacci matrix",
            "multiplicar por la misma matriz Fibonacci 2×2",
            "the two most recent sequence values",
            "los dos valores más recientes de la secuencia",
        ),
        "power": (
            "current direction vector x[t]",
            "multiply by A, then normalize",
            "multiplicar por A y después normalizar",
            "the current estimate of the dominant direction",
            "la estimación actual de la dirección dominante",
        ),
        "forecast": (
            "recent passenger history",
            "predict next month, append prediction, repeat",
            "predecir el mes siguiente, agregar la predicción y repetir",
            "the recent values used to predict the future",
            "los valores recientes usados para predecir el futuro",
        ),
    }

    state, update_en, update_es, carry_en, carry_es = examples[example]

    print("State / Estado:", state)
    print("Update EN:", update_en)
    print("Actualización ES:", update_es)
    print("Carries EN:", carry_en)
    print("Transporta ES:", carry_es)

state_output = widgets.interactive_output(
    explain_state_example,
    {"example": state_example},
)

display(widgets.VBox([state_example, state_output]))

In [ ]:
# The Fibonacci state matrix and initial state (Exercise 1, TODO 1 steps 1-2),
# rebound here so the Fibonacci explorers below run whether or not the folded
# solution was executed. The 10-step state printout, the matrix_power
# comparison, and the interpretation stay folded in the solution above.
F = np.array([[1, 1],
              [1, 0]], dtype=object)
v0 = np.array([1, 0], dtype=object)

### Interactive Fibonacci state explorer / Explorador interactivo del estado Fibonacci

Move **Steps / Pasos** or press **Play**.

Watch the two-number state:

`[next value, current value]`

grow after every repeated multiplication.

> 🇪🇸 Mueve **Pasos** o presiona **Play**.
>
> Observa cómo crece el estado de dos números:
>
> `[valor siguiente, valor actual]`

In [ ]:
#@title 🌀 Fibonacci explorer / Explorador de Fibonacci — run me / ejecútame { display-mode: 'form' }

fib_steps = widgets.IntSlider(
    value=10,
    min=1,
    max=25,
    step=1,
    description="Steps / Pasos:",
    continuous_update=False,
    style={"description_width": "100px"},
)

fib_play = widgets.Play(
    value=10,
    min=1,
    max=25,
    step=1,
    interval=600,
    description="Play",
)

# IMPORTANT FOR COLAB:
# widgets.jslink() only links values in the browser.
# The graph is redrawn by Python, so we use kernel-side widgets.link().
fib_link = widgets.link(
    (fib_play, "value"),
    (fib_steps, "value"),
)

def explore_fibonacci(steps):
    seq = []
    state = v0.copy()

    for k in range(steps + 1):
        seq.append((k, int(state[0]), int(state[1])))
        state = F @ state

    df = pd.DataFrame(
        seq,
        columns=["step", "next_value", "current_value"],
    )

    final_state = np.linalg.matrix_power(F, steps) @ v0

    plt.close("all")
    fig, ax = plt.subplots(
        figsize=(8.5, 4.2),
        constrained_layout=True,
    )

    ax.plot(
        df["step"],
        df["next_value"],
        marker="o",
        label="next value / valor siguiente",
    )
    ax.plot(
        df["step"],
        df["current_value"],
        marker="o",
        label="current value / valor actual",
    )

    # Keep the x-axis stable so the animation feels like the curve is growing.
    ax.set_xlim(0, fib_steps.max)

    # Use the largest value visible at this step, with a small margin.
    ymax = max(
        1,
        int(df[["next_value", "current_value"]].to_numpy().max())
    )
    ax.set_ylim(0, ymax * 1.12)

    ax.set_xlabel("step / paso")
    ax.set_ylabel("state value / valor del estado")
    ax.set_title(
        f"Fibonacci recursion / Recursión Fibonacci — "
        f"step/paso {steps}"
    )
    ax.legend(loc="upper left")

    plt.show()

    print("Current step / Paso actual:", steps)
    print("Current state / Estado actual:", final_state.tolist())
    print(f"Fibonacci({steps}) =", int(final_state[1]))
    print("EN: Play advances the state one update at a time.")
    print("ES: Play avanza el estado una actualización a la vez.")

# Observe the Play widget directly.
# Moving the slider also updates Play because widgets.link is bidirectional.
fib_output = widgets.interactive_output(
    explore_fibonacci,
    {"steps": fib_play},
)

display(
    widgets.VBox([
        widgets.HBox([fib_play, fib_steps]),
        fib_output,
    ])
)

### See one update numerically / Observa una actualización numéricamente

Choose a step. The notebook shows the state **before** and **after** multiplying by `F`.

> 🇪🇸 Elige un paso. El cuaderno muestra el estado **antes** y **después** de multiplicar por `F`.

In [ ]:
#@title 🔬 One Fibonacci step / Un paso de Fibonacci — run me / ejecútame { display-mode: 'form' }

fib_one_step = widgets.IntSlider(
    value=3,
    min=0,
    max=15,
    step=1,
    description="Step / Paso:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def explain_fib_step(step):
    before = np.linalg.matrix_power(F, step) @ v0
    after = F @ before

    print("Before / Antes:", before.tolist())
    print()
    print("F @ state / F @ estado")
    print(F)
    print("@")
    print(before)
    print("=")
    print(after)
    print()
    print(
        f"EN: {int(after[0])} = {int(before[0])} + {int(before[1])}; "
        f"the old current value {int(before[0])} shifts into the second position."
    )
    print(
        f"ES: {int(after[0])} = {int(before[0])} + {int(before[1])}; "
        f"el valor actual anterior {int(before[0])} pasa a la segunda posición."
    )

fib_step_output = widgets.interactive_output(
    explain_fib_step,
    {"step": fib_one_step},
)

display(widgets.VBox([fib_one_step, fib_step_output]))

## Exercise 2 — when repetition chooses a direction / Ejercicio 2 — cuando la repetición elige una dirección

Power iteration also repeats:

`x[t+1] = A @ x[t]`

but after each multiplication we normalize the vector.

For a suitable matrix, the direction tends to align with the eigenvector whose eigenvalue has the largest magnitude.

### What is an eigenvector in simple language? / ¿Qué es un autovector en lenguaje sencillo?

An eigenvector is a direction that a matrix does **not rotate away from itself**.

The matrix can stretch it, shrink it, or reverse its sign, but the direction remains special.

### Dominant eigenvector / Autovector dominante

If one eigenvalue has larger magnitude than the others, repeated multiplication tends to amplify its direction more strongly.

### Why use a synthetic 2×2 matrix here? / ¿Por qué una matriz sintética?

This exercise deliberately uses a controlled synthetic matrix because we want to change one thing cleanly:

`|λ₂ / λ₁|`


$$
x_{t} = F^{t}x_0 \approx c_1 \lambda_1^{t} v_1
\quad\text{once}\quad
\left\lvert \frac{\lambda_2}{\lambda_1} \right\rvert^{t} \text{ is small}
$$

and observe how that ratio affects convergence.

This is a mathematical experiment, not a fake real-world dataset.

> 🇪🇸 Un autovector es una dirección especial que la matriz no desvía hacia otra dirección.
>
> Usamos deliberadamente una matriz sintética `2×2` para controlar la razón `|λ₂/λ₁|` y aislar su efecto sobre la velocidad de convergencia.

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Implement power iteration.
# 2. Normalize after every multiplication.
# 3. Compare the current direction with the dominant eigenvector.
# 4. Track the angle between them.
# 5. Compare a small λ₂/λ₁ ratio with a ratio close to 1.
#
# ES:
# 1. Implementa iteración de potencias.
# 2. Normaliza después de cada multiplicación.
# 3. Compara la dirección actual con el autovector dominante.
# 4. Sigue el ángulo entre ambos.
# 5. Compara una razón λ₂/λ₁ pequeña con una cercana a 1.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def build_controlled_matrix(ratio):
    # Same eigenvectors; eigenvalues are 5 and 5*ratio.
    theta = np.pi / 6

    Q = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)],
    ])

    D = np.diag([5.0, 5.0 * ratio])

    return Q @ D @ Q.T

def dominant_direction(M):
    eigvals, eigvecs = np.linalg.eig(M)

    idx = int(np.argmax(np.abs(eigvals)))

    dominant = eigvecs[:, idx].real
    dominant /= np.linalg.norm(dominant)

    return eigvals, dominant

def power_trace(M, steps=40, seed=0):
    eigvals, dominant = dominant_direction(M)

    v = np.random.default_rng(seed).standard_normal(M.shape[0])
    v /= np.linalg.norm(v)

    rows = []

    for step in range(1, steps + 1):
        v = M @ v
        v /= np.linalg.norm(v)

        alignment = abs(float(np.dot(v, dominant)))
        alignment = np.clip(alignment, 0.0, 1.0)

        angle = np.degrees(np.arccos(alignment))
        rayleigh = float(v @ M @ v)

        rows.append({
            "step": step,
            "angle_deg": angle,
            "rayleigh": rayleigh,
            "x0": v[0],
            "x1": v[1],
        })

    return pd.DataFrame(rows), eigvals, dominant

for ratio in [0.30, 0.95]:
    M = build_controlled_matrix(ratio)
    trace, eigvals, dominant = power_trace(M)

    print("λ₂/λ₁ =", ratio)
    print("Eigenvalues / Autovalores:", np.round(np.sort(eigvals)[::-1], 3))
    print("Final angle / Ángulo final:", f"{trace['angle_deg'].iloc[-1]:.6f}°")
    print()

In [ ]:
# Reusable power-iteration helpers (Exercise 2), in a visible cell so the
# spectral-gap explorer below runs whether or not the folded solution was
# executed. The two-ratio comparison, the final-angle numbers, and the
# spectral-gap interpretation stay folded in the solution above.
def build_controlled_matrix(ratio):
    # Same eigenvectors; eigenvalues are 5 and 5*ratio.
    theta = np.pi / 6

    Q = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)],
    ])

    D = np.diag([5.0, 5.0 * ratio])

    return Q @ D @ Q.T

def dominant_direction(M):
    eigvals, eigvecs = np.linalg.eig(M)

    idx = int(np.argmax(np.abs(eigvals)))

    dominant = eigvecs[:, idx].real
    dominant /= np.linalg.norm(dominant)

    return eigvals, dominant

def power_trace(M, steps=40, seed=0):
    eigvals, dominant = dominant_direction(M)

    v = np.random.default_rng(seed).standard_normal(M.shape[0])
    v /= np.linalg.norm(v)

    rows = []

    for step in range(1, steps + 1):
        v = M @ v
        v /= np.linalg.norm(v)

        alignment = abs(float(np.dot(v, dominant)))
        alignment = np.clip(alignment, 0.0, 1.0)

        angle = np.degrees(np.arccos(alignment))
        rayleigh = float(v @ M @ v)

        rows.append({
            "step": step,
            "angle_deg": angle,
            "rayleigh": rayleigh,
            "x0": v[0],
            "x1": v[1],
        })

    return pd.DataFrame(rows), eigvals, dominant

### Interactive spectral-gap explorer / Explorador interactivo de brecha espectral

Control two things:

- `λ₂/λ₁` — how close the second eigenvalue is to the dominant one;
- number of iterations.

The left graph shows the angle to the dominant eigenvector.

The right graph shows the current direction and the dominant direction.

### Prediction / Predicción

Move `λ₂/λ₁` toward `1`.

What should happen to convergence?

The stage's [eigenvector step](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=en#eigen) runs this test on a field of arrows: every one is multiplied by `A` each frame, and the ones that kept their direction light up. Aim a vector at one and watch `A x` line up with `x`.

> 🇪🇸 Controla la razón `λ₂/λ₁` y el número de iteraciones.
>
> Acerca la razón a `1` y observa si la convergencia se vuelve más rápida o más lenta.
>
> El [paso de autovectores del escenario](https://project-delphi.github.io/tensors-workshop/interactive/linalg-stage.html?lang=es#eigen) hace esta prueba sobre un campo de flechas: cada una se multiplica por `A` en cada fotograma, y las que conservan su dirección se iluminan. Apunta un vector a una y observa cómo `A x` se alinea con `x`.


In [ ]:
#@title ⚡ Power iteration / Iteración de potencias — run me / ejecútame { display-mode: 'form' }

ratio_slider = widgets.FloatSlider(
    value=0.40,
    min=0.10,
    max=0.99,
    step=0.01,
    description="λ₂ / λ₁:",
    continuous_update=False,
    readout_format=".2f",
    style={"description_width": "80px"},
)

power_steps = widgets.IntSlider(
    value=12,
    min=1,
    max=40,
    step=1,
    description="Iterations / Iteraciones:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_power_iteration(ratio, steps):
    M = build_controlled_matrix(ratio)
    trace, eigvals, dominant = power_trace(
        M,
        steps=max(steps, 1),
        seed=0,
    )

    current = trace.iloc[-1][["x0", "x1"]].to_numpy(float)
    angle = float(trace["angle_deg"].iloc[-1])

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10.5, 4.0),
        constrained_layout=True,
    )

    axes[0].plot(
        trace["step"],
        trace["angle_deg"],
        marker="o",
    )
    axes[0].set_xlabel("iteration / iteración")
    axes[0].set_ylabel("angle to dominant direction / ángulo")
    axes[0].set_title("Convergence / Convergencia")

    axes[1].quiver(
        [0, 0],
        [0, 0],
        [dominant[0], current[0]],
        [dominant[1], current[1]],
        angles="xy",
        scale_units="xy",
        scale=1,
    )
    axes[1].set_xlim(-1.2, 1.2)
    axes[1].set_ylim(-1.2, 1.2)
    axes[1].axhline(0, linewidth=0.8)
    axes[1].axvline(0, linewidth=0.8)
    axes[1].set_aspect("equal")
    axes[1].set_title(
        "Dominant vs current direction / "
        "Dirección dominante vs actual"
    )

    plt.show()

    print("Eigenvalues / Autovalores:", np.round(np.sort(eigvals)[::-1], 3))
    print("λ₂/λ₁:", f"{ratio:.2f}")
    print("Iteration / Iteración:", steps)
    print("Angle / Ángulo:", f"{angle:.6f}°")
    print()

    if ratio > 0.85:
        print("EN: the eigenvalues are close, so the dominant direction wins slowly.")
        print("ES: los autovalores están cerca, por lo que la dirección dominante se impone lentamente.")
    else:
        print("EN: the spectral gap is larger, so the dominant direction becomes clear faster.")
        print("ES: la brecha espectral es mayor, por lo que la dirección dominante aparece más rápido.")

power_output = widgets.interactive_output(
    explore_power_iteration,
    {
        "ratio": ratio_slider,
        "steps": power_steps,
    },
)

display(
    widgets.VBox([
        widgets.HBox([ratio_slider, power_steps]),
        power_output,
    ])
)

### The important rule / La regla importante

Power iteration converges quickly when the dominant eigenvalue is clearly larger in magnitude.

A useful intuition is:

- small `|λ₂/λ₁|` → faster separation;
- `|λ₂/λ₁|` close to `1` → slower separation.

This is why the **spectral gap** matters.

> 🇪🇸 La iteración de potencias converge más rápido cuando el autovalor dominante es claramente mayor en magnitud.
>
> - `|λ₂/λ₁|` pequeño → separación más rápida;
> - `|λ₂/λ₁|` cercano a `1` → separación más lenta.

## Exercise 3 — recursive forecasting on real airline traffic / Ejercicio 3 — pronóstico recursivo con tráfico aéreo real

Now we use **144 real monthly passenger counts from 1949 to 1960**.

We hold out the final 12 months.

The model sees only the earlier months during fitting.

### Autoregressive model / Modelo autorregresivo

For a window of length `p`:

`next month = bias + w₁·old value + ... + wₚ·recent value`

$$
\widehat{y}_{t} = b + \sum_{i=1}^{p} w_i y_{t-i}
$$


The coefficients are fitted with the pseudoinverse:

`w = X⁺y`

This directly connects to Notebook 07.

### Why is this recursive? / ¿Por qué es recursivo?

After predicting one month:

1. append that prediction to history;
2. use it as an input for the next prediction;
3. repeat.

So a prediction can influence later predictions.

### Why might `p=12` make sense? / ¿Por qué puede tener sentido `p=12`?

The data are monthly. A 12-month window can contain one full annual seasonal cycle.

That does not guarantee it is the best model, but it gives the model access to one year of recent history.

> 🇪🇸 Usamos 144 observaciones mensuales reales de 1949 a 1960.
>
> Reservamos los últimos 12 meses y ajustamos el modelo solo con los meses anteriores.
>
> Cada predicción se agrega a la historia y se usa para producir la siguiente; por eso el pronóstico es recursivo.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Hold out the final 12 real months.
# 2. Fit an autoregressive model with p=12 using np.linalg.pinv.
# 3. Forecast the 12 held-out months recursively.
# 4. Compute MAPE.
# 5. Try p=3 and p=24.
# 6. Explain why prediction errors can propagate.
#
# ES:
# 1. Reserva los últimos 12 meses reales.
# 2. Ajusta un modelo autorregresivo con p=12 usando np.linalg.pinv.
# 3. Pronostica recursivamente los 12 meses reservados.
# 4. Calcula MAPE.
# 5. Prueba p=3 y p=24.
# 6. Explica por qué los errores de predicción pueden propagarse.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def fit_ar(series, p):
    rows = np.array(
        [
            series[i:i + p]
            for i in range(len(series) - p)
        ],
        dtype=float,
    )

    X_ar = np.column_stack([
        np.ones(len(rows)),
        rows,
    ])

    target = series[p:]

    return np.linalg.pinv(X_ar) @ target

def recursive_forecast(history, w_ar, p, steps):
    hist = list(
        np.asarray(
            history,
            dtype=float,
        )
    )

    out = []

    for _ in range(steps):
        recent = np.asarray(hist[-p:], dtype=float)

        nxt = float(
            w_ar[0]
            + np.dot(w_ar[1:], recent)
        )

        hist.append(nxt)
        out.append(nxt)

    return np.asarray(out)

def one_step_diagnostic(full_series, train_end, w_ar, p, steps):
    # Diagnostic only:
    # use the real previous observations when predicting each held-out month.
    # This isolates one-step model error from recursive feedback error.
    preds = []

    for t in range(train_end, min(train_end + steps, len(full_series))):
        recent_real = full_series[t - p:t]

        pred = float(
            w_ar[0]
            + np.dot(w_ar[1:], recent_real)
        )

        preds.append(pred)

    return np.asarray(preds)

def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    return np.mean(
        np.abs(predicted - actual) / actual
    )

holdout = 12

y_train = y[:-holdout]
y_test = y[-holdout:]

w12 = fit_ar(y_train, 12)

pred12_recursive = recursive_forecast(
    y_train,
    w12,
    12,
    holdout,
)

pred12_one_step = one_step_diagnostic(
    y,
    len(y_train),
    w12,
    12,
    holdout,
)

print("Training months / Meses de entrenamiento:", len(y_train))
print("Held-out months / Meses reservados:", len(y_test))
print(
    "Recursive MAPE / MAPE recursivo:",
    f"{mape(y_test, pred12_recursive):.1%}",
)
print(
    "One-step diagnostic MAPE / MAPE diagnóstico de un paso:",
    f"{mape(y_test, pred12_one_step):.1%}",
)
print()
print("EN: the one-step diagnostic uses real previous held-out values; it is not a deployable recursive forecast.")
print("ES: el diagnóstico de un paso usa valores reales previos del periodo reservado; no es un pronóstico recursivo desplegable.")

In [ ]:
# Reusable autoregressive / forecast helpers (Exercise 3), in a visible cell so
# the forecast explorers below run whether or not the folded solution was
# executed. The p=12 MAPE numbers, the p=3 / p=24 comparison, and the
# error-propagation reasoning stay folded in the solution above.
def fit_ar(series, p):
    rows = np.array(
        [
            series[i:i + p]
            for i in range(len(series) - p)
        ],
        dtype=float,
    )

    X_ar = np.column_stack([
        np.ones(len(rows)),
        rows,
    ])

    target = series[p:]

    return np.linalg.pinv(X_ar) @ target

def recursive_forecast(history, w_ar, p, steps):
    hist = list(
        np.asarray(
            history,
            dtype=float,
        )
    )

    out = []

    for _ in range(steps):
        recent = np.asarray(hist[-p:], dtype=float)

        nxt = float(
            w_ar[0]
            + np.dot(w_ar[1:], recent)
        )

        hist.append(nxt)
        out.append(nxt)

    return np.asarray(out)

def one_step_diagnostic(full_series, train_end, w_ar, p, steps):
    # Diagnostic only:
    # use the real previous observations when predicting each held-out month.
    # This isolates one-step model error from recursive feedback error.
    preds = []

    for t in range(train_end, min(train_end + steps, len(full_series))):
        recent_real = full_series[t - p:t]

        pred = float(
            w_ar[0]
            + np.dot(w_ar[1:], recent_real)
        )

        preds.append(pred)

    return np.asarray(preds)

def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    return np.mean(
        np.abs(predicted - actual) / actual
    )

### First look at the real series / Primero observa la serie real

Choose how many years of history to display.

Before fitting a model, look for:

- trend;
- repeating yearly pattern;
- increasing seasonal amplitude.

> 🇪🇸 Elige cuántos años de historia quieres ver.
>
> Antes de ajustar un modelo, busca tendencia, patrón anual repetido y cambios en la amplitud estacional.

In [ ]:
#@title ✈️ Airline history / Historial de aerolíneas — run me / ejecútame { display-mode: 'form' }

years_slider = widgets.IntSlider(
    value=12,
    min=2,
    max=12,
    step=1,
    description="Years / Años:",
    continuous_update=False,
    style={"description_width": "100px"},
)

def show_airline_history(years):
    n = years * 12

    fig, ax = plt.subplots(
        figsize=(10, 3.8),
        constrained_layout=True,
    )

    ax.plot(
        np.arange(n),
        y[:n],
        marker="o",
        markersize=3,
    )

    ax.set_xlabel("month index / índice mensual")
    ax.set_ylabel("passengers / pasajeros")
    ax.set_title(
        f"First {years} years of real airline data / "
        f"Primeros {years} años de datos reales"
    )

    plt.show()

    print("Displayed months / Meses mostrados:", n)
    print("EN: look for trend and annual seasonality before fitting a recurrence.")
    print("ES: busca tendencia y estacionalidad anual antes de ajustar una recurrencia.")

history_output = widgets.interactive_output(
    show_airline_history,
    {"years": years_slider},
)

display(widgets.VBox([years_slider, history_output]))

### Interactive recursive forecast explorer / Explorador interactivo de pronóstico recursivo

Control:

- **Window / Ventana** = how many recent months the model remembers;
- **Horizon / Horizonte** = how many recursive future steps to generate;
- **One-step diagnostic / Diagnóstico de un paso** = overlay a second line, explained below.

For the first 12 forecast steps, we have real held-out values for comparison.

After 12 months, the line moves beyond the available dataset, so there is **no real target in this notebook** for validation.

**The two lines are not the same experiment / Las dos líneas no son el mismo experimento**

In the **recursive forecast**, after predicting January the predicted January value is used to predict February. That is the deployment scenario.

In the **one-step diagnostic**, each held-out month is predicted from the **real previous values**. It is useful for diagnosis, but it is **not** the same scenario, because it uses held-out observations as lag inputs.

If the recursive error is noticeably larger than the one-step error, feedback is part of the problem.

> 🇪🇸 Controla:
>
> - **Ventana** = cuántos meses recientes recuerda el modelo;
> - **Horizonte** = cuántos pasos futuros recursivos genera;
> - **Diagnóstico de un paso** = superpone una segunda línea.
>
> Para los primeros 12 pasos existen valores reales reservados. Después de 12 meses, el pronóstico sale del conjunto disponible y **no existe un valor real en este cuaderno** para validarlo.
>
> En el pronóstico recursivo, una predicción entra en la siguiente predicción. En el diagnóstico de un paso, cada mes reservado se predice usando valores previos reales: es útil para diagnosticar, pero no representa el mismo escenario de despliegue.

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#2563eb,rgba(37,99,235,0))"></div>

Someone says the following. **Decide whether they are right before you
reveal anything** — commit to one answer, then open the check.

<div style="border-left:5px solid #2563eb;background:rgba(37,99,235,0.1);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#2563eb;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div><div style="margin:.55em 0">&ldquo;Every one-step forecast is off by about 1%, so <b>the twelve-month forecast is off by about 1% too</b>.&rdquo;</div></div>

A prediction you have committed to is worth more than one you keep
adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Alguien afirma que, como cada pronóstico a un paso se desvía cerca del 1%, <b>el pronóstico a doce meses también se desvía cerca del 1%</b>. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: does a small one-step error stay small? / Predice: ¿un error pequeño en un paso se mantiene pequeño? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo (tested in tests/test_teaching_materials.py) ---
pred_w, pred_err = 1.1, 0.01
pred_grown = pred_err * pred_w ** 12
pred_shrunk = pred_err * 0.9 ** 12

assert round(pred_grown, 4) == 0.0314
assert pred_grown > 3 * pred_err
assert pred_shrunk < pred_err
# --- end counterexample / fin del contraejemplo ---

import ipywidgets as widgets
from IPython.display import display

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown. Four bilingual answers squeezed into
# one 640px line were hard to read, and a dropdown hides three of them until
# you open it -- the wrong shape for a question whose whole point is weighing
# the options against each other. One per line, with room around them.
# Botones de opción en vez de un desplegable: una respuesta por línea.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#2563eb"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence.

    A reading wants monospace and tight rows so the numbers line up under one
    another; a sentence wants prose type and room. The two used to share one
    13px monospace column, which is most of why the reveal read as a wall.
    """
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed.

    Exactly the same words: `check_prediction` still prints, and this catches
    what it printed and gives it typography. EN and ES stay written out as
    tags rather than becoming a colour, because a reader who cannot see the
    colour still has to be able to tell the two apart.
    """
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))

pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Right — the errors stay the same size / Correcto — los errores mantienen su tamaño", "stays"),
        ("Wrong — feeding output back in compounds the error / Incorrecto — realimentar la salida compone el error", "compounds"),
        ("Wrong — the error always shrinks / Incorrecto — el error siempre se reduce", "shrinks"),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)

def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba\n      cuando quieras.".replace("\n      ", " "))
        return

    print("Start error / Error inicial:", pred_err)
    print()
    print("After 12 steps with w = 1.1 / Tras 12 pasos:", round(pred_grown, 4))
    print("After 12 steps with w = 0.9 / Tras 12 pasos:", round(pred_shrunk, 6))
    print()
    if choice == "compounds":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: a recursive forecast eats its own output, so the update rule is applied to the error as well. With w = 1.1 a 1% error is over 3% after twelve steps; with w = 0.9 it dies away. The horizon error depends on the coefficients, not on the one-step error alone.")
    print("ES: un pronóstico recursivo consume su propia salida, así que la regla de actualización se aplica también al error. Con w = 1,1 un error del 1% supera el 3% tras doce pasos; con w = 0,9 se apaga. El error a horizonte depende de los coeficientes, no solo del error a un paso.")

# The one <style> block in these notebooks, and the markdown rule does not
# cover it. ipywidgets gives no way to set the space between radio options
# from Python, and this is *widget output*, not a markdown cell: Colab strips
# <style> from markdown -- which is why every box in these notebooks is
# inline-styled -- but renders it in an output, the same path pandas' own
# Styler uses. Scoped to one added class so it can reach nothing else, and if
# it is ever dropped the options still work, just closer together.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION \u00b7 TU PREDICCI\u00d3N</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

In [ ]:
#@title 📈 Recursive forecast / Pronóstico recursivo — run me / ejecútame { display-mode: 'form' }

window_slider = widgets.IntSlider(
    value=12,
    min=3,
    max=24,
    step=1,
    description="Window / Ventana:",
    continuous_update=False,
    style={"description_width": "115px"},
)

horizon_slider = widgets.IntSlider(
    value=12,
    min=6,
    max=36,
    step=6,
    description="Horizon / Horizonte:",
    continuous_update=False,
    style={"description_width": "125px"},
)

diagnostic_toggle = widgets.Checkbox(
    value=False,
    description="One-step diagnostic / Diagnóstico de un paso",
    indent=False,
)

def explore_recursive_forecast(window, horizon, diagnostic):
    train = y[:-12]

    w_ar = fit_ar(
        train,
        window,
    )

    recursive_pred = recursive_forecast(
        train,
        w_ar,
        window,
        horizon,
    )

    train_end = len(train)

    context_start = max(
        0,
        train_end - 36,
    )

    future_idx = np.arange(
        train_end,
        train_end + horizon,
    )

    available_actual = y[
        train_end:
        min(train_end + horizon, len(y))
    ]

    actual_idx = np.arange(
        train_end,
        train_end + len(available_actual),
    )

    fig, ax = plt.subplots(
        figsize=(10.5, 4.2),
        constrained_layout=True,
    )

    ax.plot(
        np.arange(context_start, train_end),
        y[context_start:train_end],
        marker="o",
        label="training context / contexto de entrenamiento",
    )

    if len(available_actual):
        ax.plot(
            actual_idx,
            available_actual,
            marker="o",
            label="held-out actual / real reservado",
        )

    ax.plot(
        future_idx,
        recursive_pred,
        marker="o",
        label="recursive forecast / pronóstico recursivo",
    )

    one_step_pred = None

    if diagnostic and len(available_actual):
        one_step_pred = one_step_diagnostic(
            y,
            train_end,
            w_ar,
            window,
            horizon,
        )

        ax.plot(
            actual_idx,
            one_step_pred,
            marker="o",
            linestyle=":",
            label="one-step diagnostic / diagnóstico un paso",
        )

    ax.axvline(
        train_end - 0.5,
        linestyle="--",
    )

    ax.set_xlabel("month index / índice mensual")
    ax.set_ylabel("passengers / pasajeros")
    ax.set_title(
        f"Recursive forecast / Pronóstico recursivo — "
        f"window={window}, horizon={horizon}"
    )
    ax.legend()

    plt.show()

    comparable = min(
        len(available_actual),
        len(recursive_pred),
    )

    if comparable:
        err = mape(
            available_actual[:comparable],
            recursive_pred[:comparable],
        )

        print(
            f"Recursive MAPE on {comparable} real held-out months / "
            f"MAPE recursivo en {comparable} meses reales reservados: {err:.1%}"
        )

        if one_step_pred is not None:
            one_step_err = mape(
                available_actual[:comparable],
                one_step_pred[:comparable],
            )

            print(
                "One-step diagnostic MAPE / MAPE diagnóstico un paso:",
                f"{one_step_err:.1%}",
            )

    print()
    print("EN: every predicted month becomes an input to the next prediction.")
    print("ES: cada mes predicho se convierte en entrada de la siguiente predicción.")

    if one_step_pred is not None:
        print("EN: the one-step line uses real previous held-out values; use it only to diagnose recursive feedback.")
        print("ES: la línea de un paso usa valores reales previos del periodo reservado; úsala solo para diagnosticar la retroalimentación recursiva.")

    if horizon > 12:
        print("EN: after step 12, this notebook has no real future target for validation.")
        print("ES: después del paso 12, este cuaderno no tiene un valor futuro real para validar.")

forecast_output = widgets.interactive_output(
    explore_recursive_forecast,
    {
        "window": window_slider,
        "horizon": horizon_slider,
        "diagnostic": diagnostic_toggle,
    },
)

display(
    widgets.VBox([
        widgets.HBox([
            window_slider,
            horizon_slider,
        ]),
        diagnostic_toggle,
        forecast_output,
    ])
)

<details>
<summary><strong>Why can recursive error grow? / ¿Por qué puede crecer el error recursivo?</strong></summary>

Suppose the first forecast is a little too high.

The next forecast uses that slightly high value as part of its input.

If the model keeps reinforcing that mistake, the error can travel through later steps.

This does **not** mean recursive models always diverge.

It means long-horizon behaviour depends on:

- the fitted coefficients;
- the data dynamics;
- the forecast horizon;
- how much predicted information re-enters the state.

> 🇪🇸 Si la primera predicción es un poco alta, la siguiente puede usar ese valor alto como entrada.
>
> El error puede propagarse, aunque eso no significa que todo modelo recursivo necesariamente diverja.
>
> El comportamiento depende de los coeficientes, la dinámica de los datos, el horizonte y cuánta información predicha vuelve a entrar en el estado.

</details>

## Bridge to recurrent neural networks / Puente hacia redes neuronales recurrentes

The recurrence:

`x[t+1] = A @ x[t]`

is not itself an RNN.

But it gives the key mental model:

> **reuse parameters across time while carrying state forward.**

A recurrent neural network adds learned nonlinear transformations and a hidden state, but the repeated state-update idea remains.

> 🇪🇸 La recurrencia `x[t+1] = A @ x[t]` no es por sí sola una RNN.
>
> Pero entrega la idea mental central:
>
> **reutilizar parámetros a través del tiempo mientras un estado transporta información hacia adelante.**

## Quick reasoning challenge / Reto rápido de razonamiento

Choose a situation and identify what repetition is doing.

> 🇪🇸 Elige una situación e identifica qué está haciendo la repetición.

In [ ]:
#@title 🧠 Decision challenge / Reto de decisión — run me / ejecútame { display-mode: 'form' }

recursion_case = widgets.Dropdown(
    options=[
        ("Fibonacci", "fib"),
        ("Power iteration / Iteración de potencias", "power"),
        ("Recursive forecasting / Pronóstico recursivo", "forecast"),
    ],
    value="fib",
    description="Case / Caso:",
    style={"description_width": "90px"},
)

def explain_recursion_case(case):
    answers = {
        "fib": (
            "Repeated updates generate the exact sequence state.",
            "Las actualizaciones repetidas generan exactamente el estado de la secuencia.",
        ),
        "power": (
            "Repeated multiplication amplifies the dominant eigendirection.",
            "La multiplicación repetida amplifica la dirección propia dominante.",
        ),
        "forecast": (
            "Repeated prediction feeds model output back into future model input.",
            "La predicción repetida devuelve la salida del modelo como entrada futura.",
        ),
    }

    en, es = answers[case]

    print("EN:", en)
    print("ES:", es)

recursion_case_output = widgets.interactive_output(
    explain_recursion_case,
    {"case": recursion_case},
)

display(
    widgets.VBox([
        recursion_case,
        recursion_case_output,
    ])
)

## What just happened / Qué acaba de pasar

You used one idea in three different settings:

> **current state → update rule → next state → repeat**

### Four ideas to remember / Cuatro ideas para recordar

1. **State is memory. / El estado es memoria.**
2. **A recurrence reuses an update rule. / Una recurrencia reutiliza una regla de actualización.**
3. **Repeated multiplication can amplify a direction. / La multiplicación repetida puede amplificar una dirección.**
4. **Repeated prediction can propagate error. / La predicción repetida puede propagar error.**

### The sentence to remember / La frase para recordar

> **Recursion is repeated state update: the next input contains information produced by the previous step.**

> 🇪🇸
>
> **La recursión es una actualización repetida del estado: la siguiente entrada contiene información producida por el paso anterior.**


<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#2563eb,rgba(37,99,235,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **09 · Matrix factorizations / Factorizaciones matriciales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-matrix-factorizations.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)